# Multi-trial entropy, LZW & spectral clustering for FC/FCD/phFCD

Runs each model `N_TRIALS` times (different test subjects + fresh noise samples per trial) and reports **mean ± std** of:

- **Shannon entropy** of the value distribution
- **Normalized Lempel–Ziv–Welch complexity** (`c · log b(n) / (n · log b)`)
- **Adjusted Rand Index (ARI)** of spectral-clustering labels vs. the empirical target

Adapted from `wholebrain_fc_fcd_phfcd.ipynb`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'examples':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

plt.rcParams['figure.dpi'] = 120

In [ ]:
# Auto-create parent dirs for savefig (added for reproducible runs)
import matplotlib.figure as _mpl_figure
from pathlib import Path as _Path
_orig_savefig = _mpl_figure.Figure.savefig
def _patched_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, (str, _Path)):
        p = _Path(fname)
        if p.parent and not p.parent.exists():
            p.parent.mkdir(parents=True, exist_ok=True)
    return _orig_savefig(self, fname, *args, **kwargs)
_mpl_figure.Figure.savefig = _patched_savefig


## Configuration

In [ ]:
%matplotlib inline

In [ ]:
DATASET_TYPE = os.environ.get('DATASET_TYPE', 'ts_young')
DATA_PATH = 'data/ts_young/ts_young_TR0.72.mat'
CHECKPOINT_DIR = 'checkpoints'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

N_TIMEPOINTS = 120
MAX_PATHS = 2
FCD_WIN_LEN = 30
FCD_WIN_STEP = 5
N_TRIALS = 10
MODEL_FILTER = None  # e.g. {'Hopf', 'Neural SDE'}

# Sources kept across every downstream cell (summaries, plots, FC clustering).
# Set to None to keep all sources discovered in the cache.
FC_CLUSTER_FILTER = ['Target', 'Hopf', 'Hybrid Hopf', 'Neural SDE']

# Cluster counts per matrix type.
N_CLUSTERS = {'FC': 2, 'FCD': 3, 'phFCD': 3}

SAVE_DIR = PROJECT_ROOT / 'paper_new' / 'images' / 'wholebrain_complexity' / DATASET_TYPE
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = PROJECT_ROOT / '.cache' / 'wholebrain_complexity'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / f'{DATASET_TYPE}_T{N_TIMEPOINTS}_P{MAX_PATHS}_N{N_TRIALS}.pt'

FORCE_RECOMPUTE = False

print(f'Dataset    : {DATASET_TYPE}')
print(f'Device     : {DEVICE}')
print(f'Trials     : {N_TRIALS}')
print(f'Cache file : {CACHE_PATH}')
print(f'Save dir   : {SAVE_DIR}')

## Metric helpers (entropy, LZW, clustering)

In [ ]:
from src.metrics import compute_static_fc, fisher_batch_average
from src.metrics.dynamics_metrics import fcd_matrix, phfcd_matrix


def fc_average(ts):
    return fisher_batch_average(compute_static_fc(ts)).detach().cpu().numpy()


def subject_real_and_phase(ts, subject_index=0):
    s = ts[subject_index]
    return s.real.T, torch.angle(s).T


def _np(x):
    return None if x is None else x.detach().cpu().numpy()


# ---------- Shannon entropy ----------
def shannon_entropy(matrix, n_bins=32):
    if matrix is None:
        return float('nan')
    M = np.asarray(matrix)
    idx = np.triu_indices_from(M, k=1)
    vals = M[idx]
    vals = vals[np.isfinite(vals)]
    if vals.size < 2:
        return float('nan')
    bins = min(n_bins, max(4, vals.size // 4))
    vmin, vmax = float(np.nanmin(vals)), float(np.nanmax(vals))
    if vmax - vmin < 1e-12:
        return 0.0
    hist, _ = np.histogram(vals, bins=bins, range=(vmin, vmax))
    p = hist[hist > 0].astype(float)
    p /= p.sum()
    return float(-(p * np.log2(p)).sum())


# ---------- LZW complexity ----------
def lzw_codes(symbols):
    if len(symbols) == 0:
        return 0
    dictionary = {(s,): i for i, s in enumerate(set(symbols))}
    next_code = len(dictionary)
    w = ()
    n_emitted = 0
    for s in symbols:
        wc = w + (s,)
        if wc in dictionary:
            w = wc
        else:
            n_emitted += 1
            dictionary[wc] = next_code
            next_code += 1
            w = (s,)
    if w:
        n_emitted += 1
    return n_emitted


def lzw_complexity(matrix, n_bins=4):
    """Quantize upper-triangle values into n_bins equal-frequency symbols and
    return LZW complexity normalized by the Lempel-Ziv asymptotic bound."""
    if matrix is None:
        return float('nan')
    M = np.asarray(matrix)
    idx = np.triu_indices_from(M, k=1)
    vals = M[idx]
    vals = vals[np.isfinite(vals)]
    if vals.size < 4:
        return float('nan')
    edges = np.quantile(vals, np.linspace(0, 1, n_bins + 1))
    symbols = np.clip(np.digitize(vals, edges[1:-1]), 0, n_bins - 1)
    n = symbols.size
    c = lzw_codes(symbols.tolist())
    b = max(2, len(np.unique(symbols)))
    return float(c * np.log2(n) / (n * np.log2(b)))


# ---------- Spectral clustering ----------
def to_affinity(matrix):
    M = np.asarray(matrix, dtype=float)
    M = 0.5 * (M + M.T)
    if np.nanmin(M) < -1e-6:
        A = (1.0 + np.clip(M, -1.0, 1.0)) / 2.0
    else:
        A = np.clip(M, 0.0, 1.0)
    np.fill_diagonal(A, 1.0)
    return A


def cluster_labels(matrix, n_clusters, seed=0):
    if matrix is None:
        return None
    A = to_affinity(matrix)
    n = A.shape[0]
    k = min(n_clusters, n - 1)
    if k < 2:
        return np.zeros(n, dtype=int)
    sc = SpectralClustering(
        n_clusters=k,
        affinity='precomputed',
        assign_labels='kmeans',
        random_state=seed,
    )
    return sc.fit_predict(A)

## Run trials (cached)

In [ ]:
def load_test_pool_and_models():
    from src.dataset import create_data_loaders, load_dataset
    from src.models import load_model_from_checkpoint
    from src.training import HopfConfig

    cfg = HopfConfig()
    cfg.dataset_type = DATASET_TYPE
    cfg.data_path = DATA_PATH
    cfg.use_wandb = False
    cfg.dt_min = 0.05

    dataset = load_dataset(cfg, DEVICE)
    window_size = min(N_TIMEPOINTS, dataset.n_timepoints // 2)

    _, _, _, test_loader = create_data_loaders(
        dataset=dataset,
        window_size=window_size,
        batch_size=cfg.batch_size,
        n_windows_per_epoch=cfg.n_windows_per_epoch,
        train_ratio=cfg.train_ratio,
        val_ratio=cfg.val_ratio,
        seed=cfg.seed,
        device=DEVICE,
    )

    ts_chunks, ctrl_chunks = [], []
    for batch in test_loader:
        if len(batch) == 4:
            ts, _, _pid, ctrl = batch
        elif len(batch) == 3:
            ts, _, _pid = batch
            ctrl = None
        else:
            raise ValueError(f'Unexpected batch structure: {len(batch)} items')
        ts_chunks.append(ts)
        if ctrl is not None:
            ctrl_chunks.append(ctrl)
    pool_ts = torch.cat(ts_chunks, dim=0)
    pool_ctrl = torch.cat(ctrl_chunks, dim=0) if ctrl_chunks else None

    _MODEL_TAGS = [
        ('gnn_hopf', 'GNN Hopf'),
        ('hybrid_neural', 'Hybrid+Neural'),
        ('hybrid_hopf', 'Hybrid Hopf'),
        ('nsde', 'Neural SDE'),
        ('hopf', 'Hopf'),
    ]

    def short_label(stem):
        lower = stem.lower()
        for tag, label in _MODEL_TAGS:
            if tag in lower:
                return label + (' (Grid)' if 'grid' in lower else '')
        return stem

    models = {}
    for ckpt_path in sorted(Path(CHECKPOINT_DIR).glob(f'*{DATASET_TYPE}*.pt')):
        model, model_type, _ = load_model_from_checkpoint(str(ckpt_path), device=DEVICE)
        label = short_label(ckpt_path.stem)
        if model.n_rois != dataset.n_rois:
            print(f'Skipping {ckpt_path.name}: ROI mismatch ({model.n_rois} != {dataset.n_rois})')
            continue
        if MODEL_FILTER is not None and label not in MODEL_FILTER:
            continue
        models[label] = (model, model_type)
        print(f'Loaded {label:16s} ({model_type})')
    if not models:
        raise RuntimeError('No compatible checkpoints loaded.')

    return pool_ts, pool_ctrl, models


def run_trials():
    pool_ts, pool_ctrl, models = load_test_pool_and_models()
    n_pool = pool_ts.shape[0]
    print(f'\nTest pool: {n_pool} timeseries, running {N_TRIALS} trials with {MAX_PATHS} subjects each.')

    model_names = list(models.keys())
    sources = ['Target'] + model_names

    entropy_rec, lzw_rec, diff_rec, label_rec = [], [], [], {}
    fc_sum = {src: None for src in sources}
    fc_count = {src: 0 for src in sources}

    for trial in range(N_TRIALS):
        gen = torch.Generator().manual_seed(1_000 + trial)
        perm = torch.randperm(n_pool, generator=gen)[:MAX_PATHS]
        ts_target = pool_ts[perm, :, :N_TIMEPOINTS]
        control = pool_ctrl[perm] if pool_ctrl is not None else None
        initial_state = ts_target[:, :, 0]

        fc_t = fc_average(ts_target)
        tgt_real, tgt_phase = subject_real_and_phase(ts_target)
        fcd_t = _np(fcd_matrix(tgt_real, FCD_WIN_LEN, FCD_WIN_STEP))
        phfcd_t = _np(phfcd_matrix(tgt_phase))
        target_mats = {'FC': fc_t, 'FCD': fcd_t, 'phFCD': phfcd_t}
        per_src = {'Target': target_mats}

        for i, (name, (model, _)) in enumerate(models.items()):
            torch.manual_seed(7_000 + trial * 31 + i)
            kwargs = {'n_steps': N_TIMEPOINTS}
            if control is not None and getattr(model, 'n_control_dims', 0) > 0 and control.shape[-1] > 0:
                kwargs['control'] = control
            with torch.no_grad():
                ts_pred = model.forward(initial_state=initial_state, **kwargs).cpu()
            fc_p = fc_average(ts_pred)
            pred_real, pred_phase = subject_real_and_phase(ts_pred)
            fcd_p = _np(fcd_matrix(pred_real, FCD_WIN_LEN, FCD_WIN_STEP))
            phfcd_p = _np(phfcd_matrix(pred_phase))
            pred_mats = {'FC': fc_p, 'FCD': fcd_p, 'phFCD': phfcd_p}
            per_src[name] = pred_mats

            # (Target − Model) difference complexity, using the raw matrices
            # that are about to be discarded.
            for mname in ('FC', 'FCD', 'phFCD'):
                T, M = target_mats[mname], pred_mats[mname]
                diff = None if (T is None or M is None) else T - M
                diff_rec.append({
                    'trial': trial,
                    'model': name,
                    'matrix': mname,
                    'entropy': shannon_entropy(diff),
                    'lzw': lzw_complexity(diff),
                })

        for src, mats in per_src.items():
            for mname, M in mats.items():
                entropy_rec.append({'trial': trial, 'source': src, 'matrix': mname,
                                    'value': shannon_entropy(M)})
                lzw_rec.append({'trial': trial, 'source': src, 'matrix': mname,
                                'value': lzw_complexity(M)})
                label_rec[(trial, src, mname)] = cluster_labels(M, N_CLUSTERS[mname], seed=trial)
            fc_sum[src] = mats['FC'].copy() if fc_sum[src] is None else fc_sum[src] + mats['FC']
            fc_count[src] += 1
        print(f'  trial {trial + 1}/{N_TRIALS} done')

    fc_by_source_mean = {src: fc_sum[src] / fc_count[src]
                         for src in sources if fc_count[src] > 0}

    entropy_df = pd.DataFrame(entropy_rec)
    lzw_df = pd.DataFrame(lzw_rec)
    diff_df = pd.DataFrame(diff_rec)

    ari_rec = []
    for trial in range(N_TRIALS):
        for mname in N_CLUSTERS:
            t = label_rec.get((trial, 'Target', mname))
            for name in model_names:
                m = label_rec.get((trial, name, mname))
                val = (adjusted_rand_score(t, m)
                       if t is not None and m is not None and len(t) == len(m)
                       else float('nan'))
                ari_rec.append({'trial': trial, 'model': name, 'matrix': mname, 'ari': val})
    ari_df = pd.DataFrame(ari_rec)

    pair_rec = []
    for trial in range(N_TRIALS):
        for mname in N_CLUSTERS:
            for a in sources:
                for b in sources:
                    la = label_rec.get((trial, a, mname))
                    lb = label_rec.get((trial, b, mname))
                    val = (adjusted_rand_score(la, lb)
                           if la is not None and lb is not None and len(la) == len(lb)
                           else float('nan'))
                    pair_rec.append({'trial': trial, 'matrix': mname,
                                     'source_a': a, 'source_b': b, 'ari': val})
    pair_df = pd.DataFrame(pair_rec)

    return {
        'entropy_df': entropy_df,
        'lzw_df': lzw_df,
        'diff_df': diff_df,
        'ari_df': ari_df,
        'pairwise_df': pair_df,
        'sources': sources,
        'model_names': model_names,
        'fc_by_source_mean': fc_by_source_mean,
    }


EXPECTED_KEYS = {'entropy_df', 'lzw_df', 'diff_df', 'ari_df', 'pairwise_df',
                 'sources', 'model_names', 'fc_by_source_mean'}

if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    results = torch.load(CACHE_PATH, weights_only=False)
    missing = EXPECTED_KEYS - results.keys()
    if missing:
        print(f'Cache missing {missing}; recomputing...')
        results = run_trials()
        torch.save(results, CACHE_PATH)
        print(f'Refreshed cache at {CACHE_PATH}')
    else:
        print(f'Loaded cached results from {CACHE_PATH}')
else:
    print(f'Running {N_TRIALS} trials...')
    results = run_trials()
    torch.save(results, CACHE_PATH)
    print(f'Saved cache to {CACHE_PATH}')

entropy_df = results['entropy_df']
lzw_df = results['lzw_df']
diff_df = results['diff_df']
ari_df = results['ari_df']
pairwise_df = results['pairwise_df']
sources = results['sources']
model_names = results['model_names']
fc_by_source_mean = results['fc_by_source_mean']

# Apply FC_CLUSTER_FILTER globally: restrict sources, model_names, dataframes
# and FC means to the configured subset so every downstream cell stays in sync.
if FC_CLUSTER_FILTER is not None:
    keep = list(FC_CLUSTER_FILTER)
    missing = [s for s in keep if s not in sources]
    if missing:
        print(f'FC_CLUSTER_FILTER entries not in cache (ignored): {missing}')
    sources = [s for s in keep if s in sources]
    model_names = [m for m in sources if m != 'Target']
    entropy_df = entropy_df[entropy_df['source'].isin(sources)].reset_index(drop=True)
    lzw_df = lzw_df[lzw_df['source'].isin(sources)].reset_index(drop=True)
    diff_df = diff_df[diff_df['model'].isin(model_names)].reset_index(drop=True)
    ari_df = ari_df[ari_df['model'].isin(model_names)].reset_index(drop=True)
    pairwise_df = pairwise_df[
        pairwise_df['source_a'].isin(sources) & pairwise_df['source_b'].isin(sources)
    ].reset_index(drop=True)
    fc_by_source_mean = {s: fc_by_source_mean[s] for s in sources if s in fc_by_source_mean}

print(f'\nSources: {sources}')

## Aggregate (mean ± std)

In [ ]:
def summarize(long_df, value_col, group_cols):
    g = long_df.groupby(group_cols)[value_col]
    return g.agg(['mean', 'std', 'count']).reset_index()


entropy_summary = summarize(entropy_df, 'value', ['source', 'matrix'])
lzw_summary = summarize(lzw_df, 'value', ['source', 'matrix'])
ari_summary = summarize(ari_df, 'ari', ['model', 'matrix'])

print('Shannon entropy (bits) — mean ± std:')
print(entropy_summary.round(3).to_string(index=False))
print('\nLZW (normalized) — mean ± std:')
print(lzw_summary.round(3).to_string(index=False))
print('\nARI vs Target — mean ± std:')
print(ari_summary.round(3).to_string(index=False))

print('\nFC ARI vs Target — mean ± std:')
fc_ari = ari_summary[ari_summary['matrix'] == 'FC']
for _, row in fc_ari.iterrows():
    print(f"  {row['model']:16s}  {row['mean']:.3f} ± {row['std']:.3f}")

## Plots with error bars

In [ ]:
%matplotlib inline

In [ ]:
COLOR_MAP = {'FC': '#4C72B0', 'FCD': '#DD8452', 'phFCD': '#55A868'}
MATRIX_ORDER = ['FC', 'FCD', 'phFCD']


def pivot_mean_std(summary, row_key, sources_order):
    means = summary.pivot(index=row_key, columns='matrix', values='mean')
    stds = summary.pivot(index=row_key, columns='matrix', values='std').fillna(0.0)
    means = means.reindex(index=sources_order, columns=MATRIX_ORDER)
    stds = stds.reindex(index=sources_order, columns=MATRIX_ORDER)
    return means, stds


def grouped_bar_with_errors(means, stds, ax, title, ylabel):
    rows = list(means.index)
    cols = list(means.columns)
    n_groups, n_bars = len(rows), len(cols)
    bar_w = 0.8 / n_bars
    x = np.arange(n_groups)
    for i, col in enumerate(cols):
        offsets = x + (i - (n_bars - 1) / 2) * bar_w
        ax.bar(offsets, means[col].values, bar_w,
               yerr=stds[col].values, capsize=4,
               label=col, color=COLOR_MAP.get(col, f'C{i}'),
               edgecolor='black', linewidth=0.3,
               error_kw={'ecolor': 'black', 'lw': 1.0})
    ax.set_xticks(x)
    ax.set_xticklabels(rows, rotation=30, ha='right')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9, frameon=True)


fig, axes = plt.subplots(1, 2, figsize=(7, 3))
e_m, e_s = pivot_mean_std(entropy_summary, 'source', sources)
grouped_bar_with_errors(e_m, e_s, axes[0],
                        f'Shannon entropy', 'bits')
l_m, l_s = pivot_mean_std(lzw_summary, 'source', sources)
grouped_bar_with_errors(l_m, l_s, axes[1],
                        f'LZW complexity',
                        'c · log b(n) / (n · log b)')
fig.tight_layout()
out = SAVE_DIR / 'entropy_lzw_multi_trial.svg'
fig.savefig(out, bbox_inches='tight', dpi=200)
print(f'Saved {out}')
plt.show()

In [ ]:
MATRIX_SUBSET = ['FCD', 'phFCD']


def pivot_mean_std_subset(summary, row_key, sources_order, matrix_order):
    means = summary.pivot(index=row_key, columns='matrix', values='mean')
    stds = summary.pivot(index=row_key, columns='matrix', values='std').fillna(0.0)
    means = means.reindex(index=sources_order, columns=matrix_order)
    stds = stds.reindex(index=sources_order, columns=matrix_order)
    return means, stds


def grouped_bar_clean(means, stds, ax, title, ylabel):
    rows = list(means.index)
    cols = list(means.columns)
    n_groups, n_bars = len(rows), len(cols)
    bar_w = 0.8 / n_bars
    x = np.arange(n_groups)
    for i, col in enumerate(cols):
        offsets = x + (i - (n_bars - 1) / 2) * bar_w
        ax.bar(offsets, means[col].values, bar_w,
               yerr=stds[col].values, capsize=4,
               label=col, color=COLOR_MAP.get(col, f'C{i}'),
               edgecolor='black', linewidth=0.3,
               error_kw={'ecolor': 'black', 'lw': 1.0})
    ax.set_xticks(x)
    ax.set_xticklabels(rows, rotation=30, ha='right')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9, frameon=True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


fig1, ax1 = plt.subplots(figsize=(3.5, 2.2))
e_m, e_s = pivot_mean_std_subset(entropy_summary, 'source', sources, MATRIX_SUBSET)
grouped_bar_clean(e_m, e_s, ax1, 'Shannon entropy', 'bits')
fig1.tight_layout()
out1 = SAVE_DIR / 'entropy_multi_trial_fcd_phfcd.svg'
fig1.savefig(out1, bbox_inches='tight', dpi=200)
print(f'Saved {out1}')
plt.show()

fig2, ax2 = plt.subplots(figsize=(3.5, 2.2))
l_m, l_s = pivot_mean_std_subset(lzw_summary, 'source', sources, MATRIX_SUBSET)
grouped_bar_clean(l_m, l_s, ax2, 'LZW complexity', 'c · log b(n) / (n · log b)')
fig2.tight_layout()
out2 = SAVE_DIR / 'lzw_multi_trial_fcd_phfcd.svg'
fig2.savefig(out2, bbox_inches='tight', dpi=200)
print(f'Saved {out2}')
plt.show()

## Complexity of the (Target − Model) difference

Per-trial difference matrices `D = Target − Model` for FC, FCD, and phFCD,
scored with Shannon entropy and normalized LZW complexity. A low-entropy /
low-LZW difference indicates the model captures most of the structure; a
high-entropy / high-LZW difference means the residual is rich and unstructured.

The raw per-trial matrices aren't in the main cache, so this cell re-runs the
trials and stores its own cache.

In [ ]:
diff_entropy_summary = summarize(diff_df, 'entropy', ['model', 'matrix'])
diff_lzw_summary = summarize(diff_df, 'lzw', ['model', 'matrix'])

print('Shannon entropy of (Target − Model) — mean ± std:')
print(diff_entropy_summary.round(3).to_string(index=False))
print('\nLZW (normalized) of (Target − Model) — mean ± std:')
print(diff_lzw_summary.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
de_m, de_s = pivot_mean_std(diff_entropy_summary, 'model', model_names)
grouped_bar_with_errors(de_m, de_s, axes[0],
                        f'Shannon entropy of (Target − Model) (mean ± std, n={N_TRIALS})',
                        'bits')
dl_m, dl_s = pivot_mean_std(diff_lzw_summary, 'model', model_names)
grouped_bar_with_errors(dl_m, dl_s, axes[1],
                        f'LZW complexity of (Target − Model) (mean ± std, n={N_TRIALS})',
                        'c · log b(n) / (n · log b)')
fig.tight_layout()
out = SAVE_DIR / 'diff_entropy_lzw_multi_trial.png'
fig.savefig(out, bbox_inches='tight', dpi=200)
print(f'\nSaved {out}')
plt.show()

## ARI vs Target (mean ± std)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
a_m, a_s = pivot_mean_std(ari_summary, 'model', model_names)
grouped_bar_with_errors(a_m, a_s, ax,
                        f'Spectral clustering: ARI vs Target (mean ± std, n={N_TRIALS})',
                        'ARI')
ax.axhline(0, color='k', lw=0.6)
fig.tight_layout()
out = SAVE_DIR / 'ari_vs_target_multi_trial.png'
fig.savefig(out, bbox_inches='tight', dpi=200)
print(f'Saved {out}')
plt.show()

## Pairwise ARI (mean across trials, std annotated)

In [ ]:
def pairwise_mean_std(pairwise_df, matrix_name, sources):
    sub = pairwise_df[pairwise_df['matrix'] == matrix_name]
    g = sub.groupby(['source_a', 'source_b'])['ari'].agg(['mean', 'std'])
    means = g['mean'].unstack('source_b').reindex(index=sources, columns=sources)
    stds = g['std'].unstack('source_b').reindex(index=sources, columns=sources)
    return means.values, stds.values


fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, mname in zip(axes, MATRIX_ORDER):
    means, stds = pairwise_mean_std(pairwise_df, mname, sources)
    im = ax.imshow(means, vmin=-0.2, vmax=1.0, cmap='RdBu_r')
    ax.set_xticks(range(len(sources)))
    ax.set_yticks(range(len(sources)))
    ax.set_xticklabels(sources, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(sources, fontsize=8)
    ax.set_title(f'Pairwise ARI: {mname}')
    for i in range(len(sources)):
        for j in range(len(sources)):
            if np.isfinite(means[i, j]):
                txt = f'{means[i, j]:.2f}\n±{stds[i, j]:.2f}'
                col = 'white' if abs(means[i, j]) > 0.5 else 'black'
                ax.text(j, i, txt, ha='center', va='center', fontsize=6.5, color=col)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
out = SAVE_DIR / 'ari_pairwise_multi_trial.png'
fig.savefig(out, bbox_inches='tight', dpi=200)
print(f'Saved {out}')
plt.show()

## Hopf vs Hybrid Hopf FC

Mean FC (averaged across trials) for **Hopf**, **Hybrid Hopf**, and their
element-wise difference `Hybrid Hopf − Hopf`. Highlights where the hybrid
correction reshapes the static FC relative to the plain Hopf model.

In [ ]:
fc_hopf = fc_by_source_mean['Hopf']
fc_hybrid = fc_by_source_mean['Hybrid Hopf']
fc_diff = fc_hybrid - fc_hopf

diff_lim = float(np.nanmax(np.abs(fc_diff)))

fig, axes = plt.subplots(1, 3, figsize=(9, 3))

im0 = axes[0].imshow(fc_hopf, cmap='coolwarm', vmin=-1, vmax=1, interpolation='nearest')
axes[0].set_title('Hopf')
axes[0].set_xticks([]); axes[0].set_yticks([]); axes[0].set_aspect('equal')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(fc_hybrid, cmap='coolwarm', vmin=-1, vmax=1, interpolation='nearest')
axes[1].set_title('Hybrid Hopf')
axes[1].set_xticks([]); axes[1].set_yticks([]); axes[1].set_aspect('equal')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(fc_diff, cmap='coolwarm', vmin=-diff_lim, vmax=diff_lim, interpolation='nearest')
axes[2].set_title('Hybrid Hopf − Hopf')
axes[2].set_xticks([]); axes[2].set_yticks([]); axes[2].set_aspect('equal')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

fig.tight_layout()
out = SAVE_DIR / 'fc_hopf_vs_hybrid_diff.svg'
fig.savefig(out, bbox_inches='tight', dpi=200)
print(f'Saved {out}')
plt.show()

## FC graph clustering: cluster-sorted FC + co-membership adjacency

For the selected sources, spectral-cluster the **mean FC** (averaged across trials), reorder ROIs by cluster, and show:

- **Top row**: FC matrix with ROIs sorted by cluster label (block structure reveals the partition).
- **Bottom row**: binary cluster co-membership adjacency `A[i,j] = 1` iff ROI `i` and ROI `j` share a cluster.

Green lines mark cluster boundaries.

In [ ]:
# Number of clusters for this FC visualization. Override to try different K
# without touching N_CLUSTERS (which is used for the multi-trial ARI tables).
FC_PLOT_K = 2


def co_membership(labels):
    L = np.asarray(labels).reshape(-1, 1)
    return (L == L.T).astype(int)


# Sources are already filtered by FC_CLUSTER_FILTER upstream; use them directly.
available = [s for s in sources if s in fc_by_source_mean]

# Use the Target's cluster permutation for ALL panels so the block layout is
# directly comparable across sources (model panels look like Target only if
# their partition matches Target's).
target_labels = cluster_labels(fc_by_source_mean['Target'], FC_PLOT_K, seed=0)
target_order = np.argsort(target_labels, kind='stable')
target_boundaries = np.where(np.diff(target_labels[target_order]) != 0)[0] + 0.5


# Same view, but every panel reordered by the Target permutation so we can
# directly compare how each model's partition lines up with the empirical one.
fig, axes = plt.subplots(2, len(available), figsize=(1.5 * len(available), 1.5*2))
axes = np.atleast_2d(axes)

for col, src in enumerate(available):
    fc = fc_by_source_mean[src]
    labels = cluster_labels(fc, FC_PLOT_K, seed=0)
    com = co_membership(labels)

    ax_top = axes[0, col]
    ax_top.imshow(fc[np.ix_(target_order, target_order)], cmap='coolwarm',
                  vmin=-1, vmax=1, interpolation='nearest')
    ax_top.set_title(f'{src}', fontsize=10)
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.set_aspect('equal')
    for b in target_boundaries:
        ax_top.axhline(b, color='lime', lw=1.2)
        ax_top.axvline(b, color='lime', lw=1.2)

    ax_bot = axes[1, col]
    ax_bot.imshow(com[np.ix_(target_order, target_order)], cmap='gray_r',
                  vmin=0, vmax=1, interpolation='nearest')
    #ax_bot.set_title('Cluster adjacency (Target ordering)', fontsize=10)
    ax_bot.set_xticks([])
    ax_bot.set_yticks([])
    ax_bot.set_aspect('equal')
    for b in target_boundaries:
        ax_bot.axhline(b, color='lime', lw=1.2)
        ax_bot.axvline(b, color='lime', lw=1.2)

fig.tight_layout()
out = SAVE_DIR / f'fc_clustering_adjacency_target_ordered_K{FC_PLOT_K}.svg'
fig.savefig(out, bbox_inches='tight', dpi=200)
print(f'Saved {out}')
plt.show()

In [ ]:
# Number of clusters for this FC visualization. Override to try different K
# without touching N_CLUSTERS (which is used for the multi-trial ARI tables).
FC_PLOT_K = 3


def co_membership(labels):
    L = np.asarray(labels).reshape(-1, 1)
    return (L == L.T).astype(int)


# Sources are already filtered by FC_CLUSTER_FILTER upstream; use them directly.
available = [s for s in sources if s in fc_by_source_mean]

# Use the Target's cluster permutation for ALL panels so the block layout is
# directly comparable across sources (model panels look like Target only if
# their partition matches Target's).
target_labels = cluster_labels(fc_by_source_mean['Target'], FC_PLOT_K, seed=0)
target_order = np.argsort(target_labels, kind='stable')
target_boundaries = np.where(np.diff(target_labels[target_order]) != 0)[0] + 0.5


# Same view, but every panel reordered by the Target permutation so we can
# directly compare how each model's partition lines up with the empirical one.
fig, axes = plt.subplots(2, len(available), figsize=(1.5 * len(available), 1.5*2))
axes = np.atleast_2d(axes)

for col, src in enumerate(available):
    fc = fc_by_source_mean[src]
    labels = cluster_labels(fc, FC_PLOT_K, seed=0)
    com = co_membership(labels)

    ax_top = axes[0, col]
    ax_top.imshow(fc[np.ix_(target_order, target_order)], cmap='coolwarm',
                  vmin=-1, vmax=1, interpolation='nearest')
    ax_top.set_title(f'{src}', fontsize=10)
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.set_aspect('equal')
    for b in target_boundaries:
        ax_top.axhline(b, color='lime', lw=1.2)
        ax_top.axvline(b, color='lime', lw=1.2)

    ax_bot = axes[1, col]
    ax_bot.imshow(com[np.ix_(target_order, target_order)], cmap='gray_r',
                  vmin=0, vmax=1, interpolation='nearest')
    #ax_bot.set_title('Cluster adjacency (Target ordering)', fontsize=10)
    ax_bot.set_xticks([])
    ax_bot.set_yticks([])
    ax_bot.set_aspect('equal')
    for b in target_boundaries:
        ax_bot.axhline(b, color='lime', lw=1.2)
        ax_bot.axvline(b, color='lime', lw=1.2)

fig.tight_layout()
out = SAVE_DIR / f'fc_clustering_adjacency_target_ordered_K{FC_PLOT_K}.svg'
fig.savefig(out, bbox_inches='tight', dpi=200)
print(f'Saved {out}')
plt.show()